# 🧬 Séance 3 — Séquences protéiques et modèles de langage biologique (ESM-2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nevermind78/IA_Biotech/blob/main/notebooks/S3_sequences_esm2.ipynb)

**Module : Intelligence Artificielle et Biotechnologie — Atelier 3 / 5 (2h)**

> ⚙️ **Avant de commencer :** `Exécution` → `Modifier le type d'exécution` → **GPU T4**.
> (Le notebook fonctionne aussi sans GPU, mais plus lentement.)

## 🧬 Le problème biologique

Une protéine est fabriquée dans le cytoplasme, puis **adressée** vers son compartiment de travail :
noyau, mitochondrie, membrane plasmique, ou **sécrétée** hors de la cellule. Cette « adresse » est en
grande partie **écrite dans la séquence** d'acides aminés (peptide signal, séquence d'adressage
mitochondrial, hélices transmembranaires, signal de localisation nucléaire…).

> **Question :** peut-on prédire la **localisation subcellulaire** d'une protéine humaine directement
> à partir de sa séquence ? Un **modèle de langage protéique** pré-entraîné (ESM-2, Meta AI) fait-il
> mieux qu'un simple comptage des acides aminés ?

**Données :** 600 protéines humaines **Swiss-Prot** (annotées par des experts) de 4 compartiments,
150 par classe, extraites d'[UniProt](https://www.uniprot.org).

## 🎯 Objectifs de l'atelier
1. Lire des fichiers **FASTA** avec **Biopython**
2. Transformer une séquence en vecteur : **composition**, **k-mers**, puis **embeddings ESM-2**
3. Entraîner et évaluer un classifieur par **validation croisée**
4. **Comparer** objectivement les représentations : le livrable de la séance

## 🧭 Légende
▶️ exécuter · ✍️ à vous · 🔎 interpréter · 💡 solution (cellule masquée)

⏱️ **Déroulé indicatif :** FASTA & exploration (25 min) · composition & k-mers (25 min) · ESM-2 (35 min) · comparateur & interprétation (25 min) · bonus (10 min)

---
## 0. Installation et imports ▶️

In [ ]:
# ▶️ Biopython n'est pas installé par défaut sur Colab ; PyTorch et Transformers le sont.
%pip install -q biopython transformers

In [ ]:
import itertools, os, time, urllib.parse, urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from Bio import SeqIO
from transformers import AutoTokenizer, AutoModel

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sns.set_theme(style="whitegrid")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Environnement prêt — calcul sur :", DEVICE.upper())

---
## 1. Lire les fichiers FASTA ▶️

Le format **FASTA** est le format texte universel des séquences :
```
>sp|P01308|INS_HUMAN Insulin OS=Homo sapiens ...      ← ligne d'en-tête (commence par >)
MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTP...  ← séquence
```
Nous avons **un fichier par compartiment**.

In [ ]:
# ▶️ Téléchargement des 4 fichiers FASTA
LOCALISATIONS = {"noyau": "SL-0191", "mitochondrie": "SL-0173", "secretee": "SL-0243", "membrane": "SL-0039"}
BASE_GITHUB = "https://raw.githubusercontent.com/nevermind78/IA_Biotech/main/data/s3_localisation/"

def requete_uniprot(sl, n=150):
    """Plan B : même requête directement sur l'API UniProt (protéines humaines revues, 80-600 aa)."""
    autres = " ".join(f"NOT cc_scl_term:{a}" for a in ["SL-0086"] + list(LOCALISATIONS.values()) if a != sl)
    q = f"reviewed:true AND organism_id:9606 AND length:[80 TO 600] AND cc_scl_term:{sl} {autres}"
    return ("https://rest.uniprot.org/uniprotkb/search?format=fasta&size=" + str(n) + "&query="
            + urllib.parse.quote(q))

os.makedirs("fasta", exist_ok=True)
for loc, sl in LOCALISATIONS.items():
    chemin = f"fasta/{loc}.fasta"
    for url in [BASE_GITHUB + f"{loc}.fasta", requete_uniprot(sl)]:
        try:
            urllib.request.urlretrieve(url, chemin)
            print(f"✅ {loc:<13} ← {url[:70]}…")
            break
        except Exception as e:
            print(f"⚠️ {loc}: échec ({type(e).__name__}), essai suivant")

In [ ]:
# ▶️ Lecture avec Biopython : SeqIO.parse renvoie un objet SeqRecord par protéine
premiere = next(SeqIO.parse("fasta/secretee.fasta", "fasta"))
print("Identifiant :", premiere.id)
print("Description :", premiere.description)
print("Séquence    :", premiere.seq[:60], "…")
print("Longueur    :", len(premiere.seq), "acides aminés")

In [ ]:
# ▶️ Tout charger dans un DataFrame Pandas
lignes = []
for loc in LOCALISATIONS:
    for rec in SeqIO.parse(f"fasta/{loc}.fasta", "fasta"):
        lignes.append({
            "uniprot": rec.id.split("|")[1],
            "nom": rec.description.split(" ", 1)[1].split(" OS=")[0],
            "localisation": loc,
            "sequence": str(rec.seq),
            "longueur": len(rec.seq),
        })
df = pd.DataFrame(lignes)
# on écarte les rares séquences contenant des lettres non standard (U = sélénocystéine, X = inconnu)
df = df[df["sequence"].str.fullmatch("[ACDEFGHIKLMNPQRSTVWY]+")].reset_index(drop=True)
print(df["localisation"].value_counts())
df.sample(5, random_state=1)

In [ ]:
# ▶️ Distribution des longueurs par compartiment
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="localisation", y="longueur", hue="localisation", palette="Set2", legend=False)
plt.title("Longueur des protéines par compartiment")
plt.show()

---
## 2. Représentation n°1 : la composition en acides aminés

L'idée la plus simple : compter la **fréquence de chacun des 20 acides aminés** dans la séquence.
Une protéine devient un vecteur de 20 nombres (qui somment à 1).

### ✍️ Exercice 1 — Écrire la fonction `composition`
Complétez la fonction pour qu'elle renvoie la liste des 20 fréquences, dans l'ordre de `ACIDES_AMINES`.
*Indice :* `sequence.count("A")` compte les alanines ; divisez par `len(sequence)`.

In [ ]:
ACIDES_AMINES = list("ACDEFGHIKLMNPQRSTVWY")

def composition(sequence):
    # ✍️ À vous : remplacez la ligne suivante
    return None

test = composition("MKKLLA")
if test is None:
    print("⚠️ Complétez la fonction composition() puis ré-exécutez")
else:
    print(dict(zip(ACIDES_AMINES, np.round(test, 2))))

In [ ]:
#@title 💡 Solution de l'exercice 1 (exécutez-la dans tous les cas)
def composition(sequence):
    return [sequence.count(aa) / len(sequence) for aa in ACIDES_AMINES]

print(dict(zip(ACIDES_AMINES, np.round(composition("MKKLLA"), 2))))

In [ ]:
# ▶️ Matrice de composition (600 protéines × 20 acides aminés)
X_comp = np.array([composition(s) for s in df["sequence"]])
y = df["localisation"].to_numpy()
print("X_comp :", X_comp.shape)

# Composition moyenne par compartiment, comparée à la moyenne générale
comp = pd.DataFrame(X_comp, columns=ACIDES_AMINES)
comp["localisation"] = y
ecart = comp.groupby("localisation").mean() / comp[ACIDES_AMINES].mean()
plt.figure(figsize=(12, 3.5))
sns.heatmap(np.log2(ecart), cmap="RdBu_r", center=0, annot=True, fmt=".1f", cbar_kws={"label": "log2(ratio)"})
plt.title("Enrichissement (rouge) / appauvrissement (bleu) en acides aminés par compartiment")
plt.show()

### 🔎 Question 1
Quels acides aminés sont enrichis dans les protéines **membranaires** ? dans les protéines du
**noyau** ? Reliez cela à leurs propriétés chimiques (hydrophobes : `L I V F` ; chargés positivement :
`K R` qui interagissent avec l'ADN chargé négativement ; cystéines `C` des ponts disulfures des
protéines sécrétées…).

*Votre réponse :* …

### Où se trouve l'information ? Le profil N-terminal ▶️
Les signaux d'adressage vers la **sécrétion** et la **mitochondrie** sont situés au **début** (extrémité
N-terminale) de la séquence. Traçons l'hydrophobicité moyenne (échelle de Kyte-Doolittle) position par
position sur les 60 premiers acides aminés.

In [ ]:
KYTE_DOOLITTLE = {"A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5, "Q": -3.5, "E": -3.5, "G": -0.4,
                  "H": -3.2, "I": 4.5, "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6, "S": -0.8,
                  "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2}
plt.figure(figsize=(10, 4))
for loc in LOCALISATIONS:
    seqs = df.loc[df["localisation"] == loc, "sequence"]
    profil = np.array([[KYTE_DOOLITTLE[a] for a in s[:60]] for s in seqs])
    lisse = pd.Series(profil.mean(axis=0)).rolling(5, center=True, min_periods=1).mean()
    plt.plot(range(1, 61), lisse, label=loc, lw=2)
plt.axhline(0, c="k", lw=0.5)
plt.xlabel("Position dans la séquence (N-terminal →)"); plt.ylabel("Hydrophobicité moyenne")
plt.title("Profil d'hydrophobicité N-terminal par compartiment"); plt.legend()
plt.show()

🔎 Le pic hydrophobe vers les positions 5-20 des protéines **sécrétées** est la signature du
**peptide signal**, reconnu par la machinerie de sécrétion (SRP). La composition globale « dilue »
cette information locale : c'est une **limite** de cette représentation.

---
## 3. Représentation n°2 : les k-mers ▶️

Un **k-mer** est un mot de *k* lettres consécutives. Avec k = 2 (dipeptides), il y a 20² = 400 mots
possibles. Compter les k-mers capture un peu d'**ordre local** dans la séquence.

In [ ]:
def kmer_vecteur(sequence, k=2):
    """Fréquence de tous les k-mers possibles (20**k dimensions)."""
    vocabulaire = ["".join(p) for p in itertools.product(ACIDES_AMINES, repeat=k)]
    comptes = Counter(sequence[i:i + k] for i in range(len(sequence) - k + 1))
    total = max(len(sequence) - k + 1, 1)
    return [comptes[m] / total for m in vocabulaire]

print("Exemple, k=2 sur 'MKKLLA' :",
      {m: round(v, 2) for m, v in zip(["".join(p) for p in itertools.product(ACIDES_AMINES, repeat=2)],
                                      kmer_vecteur("MKKLLA", 2)) if v > 0})

X_kmer = np.array([kmer_vecteur(s, k=2) for s in df["sequence"]])
print("X_kmer :", X_kmer.shape)

---
## 4. Un protocole d'évaluation équitable ▶️

Avec seulement 600 protéines, un seul découpage apprentissage/test serait trop dépendant du hasard.
On utilise la **validation croisée à 5 plis** : chaque protéine sert 4 fois à l'apprentissage et
1 fois au test. Le **même classifieur** (régression logistique) est utilisé pour toutes les
représentations : seule la représentation change.

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluer_representation(X, nom):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1))
    res = cross_validate(clf, X, y, cv=CV, scoring=["accuracy", "f1_macro"])
    return {"représentation": nom, "dimensions": X.shape[1],
            "exactitude": res["test_accuracy"].mean(), "écart-type": res["test_accuracy"].std(),
            "F1_macro": res["test_f1_macro"].mean()}

resultats = [evaluer_representation(X_comp, "Composition (20)"),
             evaluer_representation(X_kmer, "Dipeptides k=2 (400)")]
pd.DataFrame(resultats).round(3)

### 🔎 Question 2
Le hasard donnerait 25 % d'exactitude (4 classes équilibrées). Que valent nos représentations
« simples » ? Les dipeptides (400 dimensions) font-ils beaucoup mieux que la composition (20) ?

*Votre réponse :* …

---
## 5. Représentation n°3 : les embeddings du modèle de langage ESM-2

**ESM-2** (Meta AI, 2023) est un **Transformer** entraîné sur ~65 millions de séquences protéiques à
une seule tâche : **deviner des acides aminés masqués** (comme un texte à trous). Pour y arriver, il a
dû apprendre la « grammaire » des protéines : structure, contraintes évolutives, motifs fonctionnels…

On récupère, pour chaque acide aminé, le vecteur interne du modèle (sa « compréhension » du contexte),
puis on fait la **moyenne** sur toute la séquence → un **embedding** par protéine.

| Modèle | Couches | Paramètres | Dimension | Conseillé |
|---|---|---|---|---|
| `esm2_t6_8M_UR50D` | 6 | 8 M | 320 | CPU / test rapide |
| `esm2_t12_35M_UR50D` | 12 | 35 M | 480 | **GPU T4 (défaut)** |
| `esm2_t30_150M_UR50D` | 30 | 150 M | 640 | GPU, meilleur |

In [ ]:
# ✍️ Choix du modèle (commencez par 35M ; essayez 8M et 150M au défi 1)
MODELE_ESM = "facebook/esm2_t12_35M_UR50D"  #@param ["facebook/esm2_t6_8M_UR50D", "facebook/esm2_t12_35M_UR50D", "facebook/esm2_t30_150M_UR50D"]

tokenizer = AutoTokenizer.from_pretrained(MODELE_ESM)
modele = AutoModel.from_pretrained(MODELE_ESM).to(DEVICE).eval()
print(f"✅ {MODELE_ESM} chargé : {sum(p.numel() for p in modele.parameters())/1e6:.0f} M paramètres")

In [ ]:
# ▶️ Ce que « voit » le modèle : une séquence découpée en jetons (tokens)
exemple = "MKTAYIAK"
enc = tokenizer(exemple, return_tensors="pt")
print("Jetons      :", tokenizer.convert_ids_to_tokens(enc["input_ids"][0]))
print("Identifiants:", enc["input_ids"][0].tolist())
with torch.no_grad():
    sortie = modele(**enc.to(DEVICE)).last_hidden_state
print("Sortie      :", tuple(sortie.shape), "= (1 séquence, 8 acides aminés + 2 jetons spéciaux, dimension)")

In [ ]:
# ▶️ Fonction fournie : embeddings moyens (mean pooling) d'une liste de séquences
@torch.no_grad()
def embeddings_esm(sequences, taille_lot=8, longueur_max=1022):
    ordre = np.argsort([len(s) for s in sequences])          # trier par longueur = moins de padding
    resultats = [None] * len(sequences)
    for debut in range(0, len(sequences), taille_lot):
        idx = ordre[debut:debut + taille_lot]
        lot = [sequences[i][:longueur_max] for i in idx]
        enc = tokenizer(lot, return_tensors="pt", padding=True).to(DEVICE)
        h = modele(**enc).last_hidden_state                   # (lot, longueur, dimension)
        masque = enc["attention_mask"].clone()
        masque[:, 0] = 0                                       # retire le jeton <cls>
        masque[torch.arange(len(lot)), enc["attention_mask"].sum(1) - 1] = 0   # retire <eos>
        moyenne = (h * masque.unsqueeze(-1)).sum(1) / masque.sum(1, keepdim=True)
        for i, v in zip(idx, moyenne.float().cpu().numpy()):
            resultats[i] = v
    return np.vstack(resultats)

t0 = time.time()
X_esm = embeddings_esm(df["sequence"].tolist())
print(f"X_esm : {X_esm.shape} — calculé en {time.time() - t0:.0f} s")

In [ ]:
# ▶️ Évaluation avec exactement le même protocole
resultats.append(evaluer_representation(X_esm, f"ESM-2 {MODELE_ESM.split('_')[2]} ({X_esm.shape[1]})"))
tableau = pd.DataFrame(resultats).round(3)
tableau

---
## 6. 🏁 Livrable : le comparateur ▶️

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
couleurs = ["#9e9e9e", "#64b5f6", "#e65100", "#ad1457", "#2e7d32"][:len(tableau)]
barres = ax.barh(tableau["représentation"], tableau["exactitude"], xerr=tableau["écart-type"],
                 color=couleurs, capsize=4)
ax.axvline(0.25, ls="--", c="k", label="Hasard (25 %)")
ax.bar_label(barres, labels=[f"{v:.0%}" for v in tableau["exactitude"]], padding=8)
ax.set_xlim(0, 1); ax.set_xlabel("Exactitude moyenne (validation croisée 5 plis)")
ax.set_title("Prédiction de la localisation subcellulaire : comparaison des représentations")
ax.legend(loc="lower right"); ax.invert_yaxis()
plt.tight_layout(); plt.savefig("comparateur_representations.png", dpi=120); plt.show()

In [ ]:
# ▶️ Matrices de confusion : composition vs ESM-2
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (X, nom) in zip(axes, [(X_comp, "Composition"), (X_esm, "ESM-2")]):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1))
    y_pred = cross_val_predict(clf, X, y, cv=CV)
    ConfusionMatrixDisplay.from_predictions(y, y_pred, ax=ax, cmap="Blues", colorbar=False,
                                            normalize="true", values_format=".0%")
    ax.set_title(nom)
plt.tight_layout(); plt.show()

### 🔎 Question 3
1. Quel est le **gain** d'exactitude d'ESM-2 par rapport à la composition ?
2. Quelles classes sont le plus souvent **confondues** ? Est-ce biologiquement logique ?
   (pensez aux protéines membranaires… qui ont aussi un peptide signal !)
3. ESM-2 n'a **jamais vu d'étiquette de localisation** pendant son entraînement. Comment
   expliquer qu'il soit si bon ?

*Votre réponse :* …

In [ ]:
# ▶️ Visualiser l'espace des embeddings (t-SNE) : les compartiments forment-ils des groupes ?
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (X, nom) in zip(axes, [(X_comp, "Composition"), (X_esm, "ESM-2")]):
    X2 = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(
        PCA(n_components=min(50, X.shape[1]), random_state=42).fit_transform(StandardScaler().fit_transform(X)))
    sns.scatterplot(x=X2[:, 0], y=X2[:, 1], hue=y, s=15, ax=ax, palette="Set2")
    ax.set_title(f"t-SNE — {nom}"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

---
## 7. Prédire la localisation de nouvelles protéines ✍️

In [ ]:
# ✍️ Ajoutez des identifiants UniProt de votre choix (https://www.uniprot.org)
NOUVELLES = {
    "P01308": "Insuline",
    "P62805": "Histone H4",
    "P13073": "Cytochrome c oxydase sous-unité 4",
    "P01730": "Récepteur CD4",
}

clf_final = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1)).fit(X_esm, y)
seqs_new = []
for acc in NOUVELLES:
    with urllib.request.urlopen(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta") as r:
        seqs_new.append("".join(r.read().decode().splitlines()[1:]))
proba = clf_final.predict_proba(embeddings_esm(seqs_new))
pred = pd.DataFrame(proba, columns=clf_final.classes_, index=[f"{v} ({k})" for k, v in NOUVELLES.items()]).round(2)
pred["prédiction"] = pred.idxmax(axis=1)
pred["déjà_vue_en_apprentissage"] = [k in set(df["uniprot"]) for k in NOUVELLES]
pred

---
## 🚀 Défis (si vous avez le temps)

**Défi 1 — Taille du modèle.** Changez `MODELE_ESM` (8M puis 150M) et ré-exécutez à partir de la
section 5. Le gain est-il proportionnel au nombre de paramètres ? Notez le temps de calcul.

**Défi 2 — Où est l'information ?** Calculez les embeddings ESM-2 en ne gardant que les **50 premiers**
acides aminés, puis les **50 derniers**. Lequel prédit le mieux la localisation ? Pourquoi ?

In [ ]:
#@title 💡 Solution du défi 2
X_debut = embeddings_esm([s[:50] for s in df["sequence"]])
X_fin = embeddings_esm([s[-50:] for s in df["sequence"]])
defi2 = pd.DataFrame([evaluer_representation(X_debut, "ESM-2 : 50 premiers aa (N-term)"),
                      evaluer_representation(X_fin, "ESM-2 : 50 derniers aa (C-term)"),
                      evaluer_representation(X_esm, "ESM-2 : séquence entière")]).round(3)
defi2

---
## 📝 Bilan de la séance

| Étape | Notion clé |
|---|---|
| FASTA + Biopython | format universel des séquences, `SeqIO.parse` |
| Composition, k-mers | **feature engineering** manuel : simple, interprétable, perd l'ordre |
| ESM-2 | **modèle de langage** auto-supervisé, **embeddings**, **apprentissage par transfert** |
| Validation croisée | comparaison **équitable** de représentations |

⚠️ **Limite importante :** des protéines **homologues** (de la même famille) peuvent se retrouver à la
fois dans l'apprentissage et dans le test, ce qui **surestime** les performances. Les études sérieuses
regroupent d'abord les séquences par similarité (ex. < 30 % d'identité avec MMseqs2/CD-HIT).

**Livrable à rendre :** la figure `comparateur_representations.png` et vos réponses aux questions 🔎 1 à 3.

➡️ **Séance suivante :** l'IA pour l'**imagerie cellulaire** (réseaux de neurones convolutifs, Cellpose).